In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
from astroquery.gaia import Gaia
from concurrent.futures import ThreadPoolExecutor, as_completed

size = [0.2000,0.1650]
for s in size:
    # Galactic latitude array
    b_values = np.arange(-6, 5, s)
    for i in b_values: # remove high extinction latitudes
        if i > -2.2 and i < 2.1:
            index = np.where(b_values==i)
            b_values = np.delete(b_values, index)

    # Galactic longitude array
    l_values = np.arange(-9.9, 10, s)
    # Original longitudes are preserved
    l_original = np.copy(l_values)
    # Convert negative longitudes to 360+ for Gaia queries
    l_query = np.array([l if l >= 0 else 360 + l for l in l_values])

    # Rectangular patch size (degrees)
    delta_l = s
    delta_b = s

    def download_gaia_tile(l_query_val, l_original_val, b_val, delta_l=0.2475, delta_b=0.2475):
        """Downloads Gaia DR3 data for a rectangular patch around (l, b)."""
        l_min = l_query_val - delta_l/2
        l_max = l_query_val + delta_l/2
        b_min = b_val - delta_b/2
        b_max = b_val + delta_b/2

        # Use original l value in the filename
        l_filename = l_original_val
        dir_name = f'Gaia_Data/b_{b_val:0.4f}_{delta_b:0.4f}_degrees'
        os.makedirs(dir_name, exist_ok=True)
        csv_path = Path(dir_name) / f'l_{l_filename:0.4f}_b_{b_val:0.4f}.csv'

        query = f"""
        SELECT phot_g_mean_mag, bp_rp, parallax, parallax_error, ra, dec, l, b, ruwe, ag_gspphot
        FROM gaiadr3.gaia_source
        WHERE l BETWEEN {l_min} AND {l_max}
        AND b BETWEEN {b_min} AND {b_max}
        AND ruwe < 1.4
        AND phot_g_mean_mag < 20.5
        AND parallax IS NOT NULL
        AND parallax_error IS NOT NULL
        AND bp_rp IS NOT NULL
        AND phot_g_mean_mag IS NOT NULL
        ORDER BY l, b
        """
        try:
            job = Gaia.launch_job_async(query)
            results = job.get_results()
            df = results.to_pandas()
            df.to_csv(csv_path, index=False)
            print("Downloaded:", csv_path)
        except Exception as e:
            print(f"Failed l={l_filename}, b={b_val}: {e}")

    # Maximum number of threads
    max_threads = 8

    # Loop over b values and download patches in parallel
    for b_val in b_values:
        tiles = list(zip(l_query, l_original))
        with ThreadPoolExecutor(max_workers=max_threads) as executor:
            futures = [executor.submit(download_gaia_tile, lq, lo, b_val, delta_l, delta_b) 
                    for lq, lo in tiles]
            for future in as_completed(futures):
                future.result()  # raise any exceptions

The archive is unstable and may perform below expectations. Please avoid launching intense Python query showers. Please contact the Gaia helpdesk in case of questions (https://www.cosmos.esa.int/web/gaia/gaia-helpdesk). Workaround solutions for the issues following the recent infrastructure upgrade: https://www.cosmos.esa.int/web/gaia/news#WorkaroundArchive
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
INFO: Query finished. [astroquery.utils.tap.core]
Downloaded: Gaia_Data\b_-6.0000_0.2000_degrees\l_-9.3000_b_-6.0000.csv
Downloaded: Gaia_Data\b_-6.0000_0.2000_degrees\l_-9.7000_b_-6.0000.csv
Downloaded: Gaia_Data\b_-6.0000_0.2000_degrees\l_-9.5000_b_-6.0000.csv
Downloaded: Gaia_Data\b_-6.

In [ ]:
import os
from pathlib import Path
import numpy as np
import pandas as pd
np.set_printoptions(legacy = "1.25")
import fitting_routines as fit
import plot_functions as plot
import warnings
warnings.filterwarnings("ignore") # none of the warnings break anything

size = [0.2475,0.200] # size of sightline FOV

COLUMNS = ["Latitude", "Long", "Mean Parallax", "Mean Parallax Error", "Parallax Dispersion", 
            "Number at parallax mean", "Clump 1 Parallax", "Clump 2 Parallax", "Clump 1 Parallax Sigma",
            "Clump 2 Parallax Sigma", "Number at Clump 1 Parallax Mean", "Number at Clump 2 Parallax Mean",
            "Mean Magnitude", "Magnitude Dispersion", "Number at Mag mean", 
            "Mean Color", "Color Dispersion", "Number at color mean", "Count", "Magnitude Distinctness", 
            "D1Mag Distinctness", "D2Mag Distinctness", "Double Clump width", "Color Distinctness", 
            "Distance Modulus", "Red Clump Fraction", "Red Clump Width", "Red Clump 1 Width",
            "Red Clump 2 Width"]

for s in size:
    TotalGood_Single = pd.DataFrame(columns=[COLUMNS])
    TotalGood_Double = pd.DataFrame(columns=[COLUMNS])
    TotalBad = pd.DataFrame(columns=[COLUMNS])

    b = np.arange(-6,5,s)
    Al = np.arange(-9.9,10,s)

    for i in b: # run functions for sightlines
        # find folder with csvs
        dir_name = f'Gaia_Data/b_{i:0.4f}_{s:0.4f}_degrees'
        os.makedirs(dir_name, exist_ok=True)
        path = Path(dir_name)

        #create folders and lists for plots
        plot_path = f'Plots/Diagnostics/b_{i:0.4f}_diagnostics_test_{s:0.4f}_degrees'
        os.makedirs(plot_path, exist_ok=True)
        plotpath = Path(plot_path)

        Good_Single = pd.DataFrame(columns=[COLUMNS])
        Good_Double = pd.DataFrame(columns=[COLUMNS])
        Bad = pd.DataFrame(columns=[COLUMNS])
        count = 0
        for l in Al:
            try:
                df = pd.read_csv(path / f'l_{l:0.4f}_b_{i:0.4f}.csv') # read in csv
                df["l"] = df["l"]-360
                #df = df.loc[(df["b"]<i)&(df["l"]<l)]
                redclump = fit.redclumpfinder_v1(df) # find the red clump
                zeropoint = fit.zeropoint(redclump)
                parallax = fit.parallax(redclump, zeropoint) # sigma clip parallax both before and after applying zero point correction
                doubleclump = fit.doubleclump_finder_v1(df, redclump)
                doubleparallax = fit.doublefinalfit(redclump, zeropoint)
                plot.DRedClumpPlot(redclump, parallax, df, doubleclump, doubleparallax, plotpath=plotpath, l=l, b=i, s=s)
                #print("Finished fitting", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                count = count + 1
            except RuntimeError: # sometimes it will fail to fit due to the post-histogram selection and will crash
                try:
                    redclump = fit.redclumpfinder_v2(df) # see if it will fit if we change the spread for the first cut
                    zeropoint = fit.zeropoint(redclump)
                    parallax = fit.parallax(redclump, zeropoint)
                    doubleclump = fit.doubleclump_finder_v2(df, redclump)
                    doubleparallax = fit.doublefinalfit(redclump, zeropoint)
                    plot.DRedClumpPlot(redclump, parallax, df, doubleclump, doubleparallax, plotpath=plotpath, l=l, b=i, s=s)
                    #print("Finished fitting", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                    count = count + 1
                except RuntimeError:
                    #print("Could not fit", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}') # if not, save the fitting cut and 2D histogram for troubleshooting
                    plot.DRedClumpPlot_runtime(redclump, df, plotpath=plotpath, l=l, b=i, s=s)
                    continue
                except KeyError: # usually means it could find a fit, but for some reason could not go beyond plotting the histograms
                    #print("Key Error for", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                    plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                    continue
                except TypeError: # error with parameters, could not make a fit
                    #print("Type Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                    plot.DRedClumpPlot_type(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                    continue
                except ValueError: # error with parameters, could not make a fit
                    plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                    #print("Value Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                    continue
                except NameError:
                    plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                    #print("Value Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                    continue
                continue
            except KeyError: # usually means it could find a fit, but for some reason could not go beyond plotting the histograms
                #print("Key Error for", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                continue
            except FileNotFoundError: # sometimes the data downloader cannot find a sightline along each latitude and so various
                                     # sightlines may, incosistently, not be available in a latitude
              #print("Could not find file", f'l_{l:0.4f}_b_{i:0.4f}')
              continue
            except TypeError: # error with parameters, could not make a fit
                #print("Type Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                plot.DRedClumpPlot_type(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                continue
            except ValueError: # error with parameters, could not make a fit
                plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                #print("Value Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                continue
            except NameError:
                plot.DRedClumpPlot_break(redclump, df, doubleclump, plotpath=plotpath, l=l, b=i, s=s)
                #print("Value Error for:", f'l_{l:0.4f}_b_{i:0.4f}_s_{s:0.4f}')
                continue

            # do not modify placement of these; concatenation needs to happen after all of the try/except loops or else
            # the columns can have different lengths causing an error when analyzing
            if redclump["Magnitude Distinctness"]>1 and redclump["Color Distinctness"]>1 and redclump["Red Clump Width"] < .45 and redclump["Red Clump Width"]>0.29 and np.abs(parallax["parallax sigma"])<0.3 and (doubleclump["peak difference"] > 0.65 or doubleclump["clump 1 distinctness"]<1 or doubleclump["clump 2 distinctness"]<1):
                fitcut = parallax["dataframe"]
                Good_Single = pd.concat([pd.DataFrame([[i, l, parallax["parallax peak"], parallax["mean error"], np.abs(parallax["parallax sigma"]),
                                                parallax["Number at parallax mean"], doubleparallax["parallax peak 1"],
                                                doubleparallax["parallax peak 2"], doubleparallax["parallax sigma 1"],
                                                doubleparallax["parallax sigma 2"], doubleparallax["Number at parallax mean 1"],
                                                doubleparallax["Number at parallax mean 2"], redclump["mag"], redclump["mag sigma"], 
                                                redclump["Number at mag mean"], redclump["color"], redclump["color sigma"], 
                                                redclump["Number at color mean"], len(fitcut["phot_g_mean_mag"]), 
                                                redclump["Magnitude Distinctness"], doubleclump["clump 1 distinctness"],
                                                doubleclump["clump 2 distinctness"], doubleclump["peak difference"], 
                                                redclump["Color Distinctness"], 5*np.log10(1/parallax["parallax peak"])-5, 
                                                redclump["Red Clump Fraction"], redclump["Red Clump Width"],
                                                doubleclump["Red Clump 1 Width"], doubleclump["Red Clump 2 Width"]]],
                                                columns=Good_Single.columns), Good_Single], ignore_index=True)
            
            if redclump["Magnitude Distinctness"]>1 and redclump["Color Distinctness"]>1 and redclump["Red Clump Width"] < .45 and redclump["Red Clump Width"]>0.29 and np.abs(parallax["parallax sigma"])<0.3 and doubleclump["peak difference"] < 0.65 and doubleclump["clump 1 distinctness"]>1 and doubleclump["clump 2 distinctness"]>1:
                fitcut = parallax["dataframe"]
                Good_Double = pd.concat([pd.DataFrame([[i, l, parallax["parallax peak"], parallax["mean error"], np.abs(parallax["parallax sigma"]),
                                                parallax["Number at parallax mean"], doubleparallax["parallax peak 1"],
                                                doubleparallax["parallax peak 2"], doubleparallax["parallax sigma 1"],
                                                doubleparallax["parallax sigma 2"], doubleparallax["Number at parallax mean 1"],
                                                doubleparallax["Number at parallax mean 2"], redclump["mag"], redclump["mag sigma"], 
                                                redclump["Number at mag mean"], redclump["color"], redclump["color sigma"], 
                                                redclump["Number at color mean"], len(fitcut["phot_g_mean_mag"]), 
                                                redclump["Magnitude Distinctness"], doubleclump["clump 1 distinctness"],
                                                doubleclump["clump 2 distinctness"], doubleclump["peak difference"], 
                                                redclump["Color Distinctness"], 5*np.log10(1/parallax["parallax peak"])-5, 
                                                redclump["Red Clump Fraction"], redclump["Red Clump Width"],
                                                doubleclump["Red Clump 1 Width"], doubleclump["Red Clump 2 Width"]]],
                                                columns=Good_Double.columns), Good_Double], ignore_index=True)

            else:
                Badfitcut = parallax["dataframe"]
                Bad = pd.concat([pd.DataFrame([[i, l, parallax["parallax peak"], parallax["mean error"], np.abs(parallax["parallax sigma"]),
                                                parallax["Number at parallax mean"], doubleparallax["parallax peak 1"],
                                                doubleparallax["parallax peak 2"], doubleparallax["parallax sigma 1"],
                                                doubleparallax["parallax sigma 2"], doubleparallax["Number at parallax mean 1"],
                                                doubleparallax["Number at parallax mean 2"], redclump["mag"], redclump["mag sigma"], 
                                                redclump["Number at mag mean"], redclump["color"], redclump["color sigma"], 
                                                redclump["Number at color mean"], len(Badfitcut["phot_g_mean_mag"]), 
                                                redclump["Magnitude Distinctness"], doubleclump["clump 1 distinctness"],
                                                doubleclump["clump 2 distinctness"], doubleclump["peak difference"], 
                                                redclump["Color Distinctness"], 5*np.log10(1/parallax["parallax peak"])-5, 
                                                redclump["Red Clump Fraction"], redclump["Red Clump Width"],
                                                doubleclump["Red Clump 1 Width"], doubleclump["Red Clump 2 Width"]]],
                                                columns=Bad.columns), Bad], ignore_index=True)
            
        Good_Single.to_csv(path / f"b_{i:0.4f}_s_{s:0.4f}_good_single_cuts.csv") # save all cuts along a latitude to a csv so you can add missed bad sightlines 
                                                        # to bad csv
        Good_Double.to_csv(path / f"b_{i:0.4f}_s_{s:0.4f}_good_double_cuts.csv")
        Bad.to_csv(path / f"b_{i:0.4f}_s_{s:0.4f}_bad_cuts.csv")

        TotalGood_Single = pd.concat([TotalGood_Single,Good_Single], ignore_index=True)
        TotalGood_Double = pd.concat([TotalGood_Double,Good_Double], ignore_index=True)
        TotalBad  = pd.concat([TotalBad,Bad], ignore_index=True)

        try:
            plot_path2 = f'Plots/mean plots'
            os.makedirs(plot_path2, exist_ok=True)
            plotpath2 = Path(plot_path2)
            meanplot = pd.read_csv(path / f"b_{i:0.4f}_s_{s:0.4f}_good_single_cuts.csv")
            plot.Dmeanplot(meanplot, Al=Al, b=i, plotpath=plotpath2, s=s)
            #print("Finished", f'{i:0.4f}')
        except:
            print(f"nothing to plot for (b = {i:0.4f}, s = {s:0.4f}")
            continue
    TotalGood_Single
    TotalGood_Double
    TotalBad
    TotalGood_Single.to_csv(f'Gaia_Data/Total_Good_Single_Fits_s_{s:0.4f}')
    TotalGood_Double.to_csv(f'Gaia_Data/Total_Good_Double_Fits_s_{s:0.4f}')
    TotalBad.to_csv(f'Gaia_Data/Total_Bad_Fits_s_{s:0.4f}')
    plot_path3 = f'Plots'
    os.makedirs(plot_path3, exist_ok=True)
    plotpath3 = Path(plot_path3)
    plot.Dtotalplot(TotalGood_Single,plotpath=plotpath3, s=s)
    print(f"Percent bad sightlines for sightline size {s:0.4f}:", len(TotalBad["Mean Parallax"])/(len(b)*len(Al))*100, "%")